<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/ppo_bootstrap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Force headless pygame BEFORE anything imports pygame (directly or
# transitively). SDL picks its video driver at import time, so
# setting this later is a no-op for an already-loaded SDL backend.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

# Reinforce Tactics — PPO Bootstrap Curriculum

Warm-start a `MaskablePPO` policy by walking it through a fixed curriculum of
`(map, opponent)` stages before handing the resulting checkpoint off to
self-play (`notebooks/ppo_training.ipynb`).

Six stages: starter map (random → simple → medium), then beginner map
(random → simple → medium). A stage advances only when its eval win-rate
stays at or above its `promotion_win_rate` for `patience` consecutive
evaluations. If a stage exhausts its `max_timesteps` budget without
promoting, the runner raises `CurriculumStalled` — fix the underlying
issue (reward shaping, hyperparams) before raising the budget.

Stage list lives in `configs/ppo/bootstrap.yaml`; edit there to tune
thresholds, budgets, or PPO hyperparams without touching this notebook.

**Runtime:** GPU recommended (L4 or better). CPU is possible but slower.

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3 sb3-contrib tensorboard pandas numpy torch matplotlib opencv-python-headless

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repo and install as a package
import os
import sys
from pathlib import Path

REPO_DIR = Path("reinforce-tactics")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
elif Path("notebooks").exists():
    # Already inside the repo
    os.chdir("..")
else:
    print("Cloning repository...")
    !git clone https://github.com/kuds/reinforce-tactics.git
    os.chdir(REPO_DIR)

# Install the package so all imports resolve
!pip install -q -e .

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

## 2. Imports

In [ ]:
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
from sb3_contrib import MaskablePPO

from reinforcetactics.rl.bootstrap import CurriculumStalled, run_curriculum
from reinforcetactics.rl.config import load_config
from reinforcetactics.rl.evaluation import evaluate_model
from reinforcetactics.rl.masking import make_maskable_env

print("All imports successful.")

## 2b. Storage configuration

Choose where to save outputs. Set `USE_GOOGLE_DRIVE = True` to persist
results to Google Drive (recommended for Colab — files survive runtime
disconnects). Set `False` to use local/Colab storage.

Each execution saves outputs under a datetime-stamped subfolder
(e.g. `benchmarks/bootstrap/20250615_143022/`), so previous runs are
preserved automatically.

In [ ]:
USE_GOOGLE_DRIVE = True
DRIVE_BASE_DIR = "reinforce-tactics/benchmarks"

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        SAVE_DIR = Path("/content/drive/MyDrive") / DRIVE_BASE_DIR
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Google Drive mounted successfully.")
        print(f"Save directory: {SAVE_DIR}")
    except ImportError:
        print("WARNING: google.colab not available (not running in Colab).")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
    except Exception as e:
        print(f"WARNING: Failed to mount Google Drive: {e}")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
else:
    SAVE_DIR = None
    print("Using local storage (default).")
    print("  Tip: Set USE_GOOGLE_DRIVE = True to persist results to Google Drive.")

## 3. Configuration

Reads `configs/ppo/bootstrap.yaml`. To experiment, copy that file, edit, and
point `CONFIG_PATH` at the copy.

Each stage can override `max_steps`, `max_turns`, and `ent_coef` to
handle map-size or exploration shifts (the bigger `beginner.csv` map
needs a longer turn budget; the first beginner stage gets an entropy
bump to break the starter-map policy out of its rut). The summary
table below highlights any active overrides per stage; absent values
inherit `cfg.env` / `cfg.ppo` defaults.

In [ ]:
CONFIG_PATH = Path("configs/ppo/bootstrap.yaml")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
if SAVE_DIR is not None:
    OUTPUT_DIR = SAVE_DIR / "bootstrap" / RUN_ID
else:
    OUTPUT_DIR = Path("benchmarks") / "bootstrap" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Single ``charts/`` root for every PNG produced by this run, mirroring
# the layout used by ppo_training.ipynb so downstream tooling that scans
# benchmark dirs sees a consistent structure across both notebooks.
CHARTS_DIR = OUTPUT_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Snapshot the source YAML into the run dir so this exact curriculum
# is reproducible even if configs/ppo/bootstrap.yaml is edited later.
# run_curriculum writes a per-stage ``config.json`` with the resolved
# post-override values; this copy preserves comments and the original
# pre-override view that was fed to ``load_config``.
import shutil

shutil.copy2(CONFIG_PATH, OUTPUT_DIR / CONFIG_PATH.name)

cfg = load_config(CONFIG_PATH)

# Hard-code the available unit roster to the first 5 unit types
# (W, M, C, A, K). ``env.enabled_units`` is global -- it applies to
# training envs, the section-6 sanity eval, and the section-8 replay
# envs alike.
# cfg = apply_overrides(cfg, {"env.enabled_units": ["W", "M", "C", "A", "K"]})

print(f"Config:        {CONFIG_PATH}")
print(f"enabled_units: {cfg.env.enabled_units}")
print(f"Defaults:      max_steps={cfg.env.max_steps}, max_turns={cfg.env.max_turns}, ent_coef={cfg.ppo.ent_coef}")
print(f"Stages:        {len(cfg.curriculum.stages)}")

# Per-stage table -- highlight per-stage overrides so the curriculum's
# map-transition handling (longer turn budget, entropy bump, reward
# reshape) is visible at a glance. Column widths are computed from the
# actual stage / opponent names (with reasonable minimums) so a long
# name like ``skirmish_mixed_medium_advanced`` doesn't wrap or push
# subsequent columns out of alignment.
name_col_w = max(26, *(len(s.name) for s in cfg.curriculum.stages))
opp_col_w = max(16, *(len(s.opponent) for s in cfg.curriculum.stages))
header = (
    f"  {'name':<{name_col_w}s}  {'opp':<{opp_col_w}s}  {'WR>=':>5s}  "
    f"{'budget':>10s}  {'max_steps':>10s}  {'max_turns':>10s}  "
    f"{'ent_coef':>13s}  {'reward keys overridden'}"
)
print(header)
print("  " + "-" * (len(header) - 2))


def _fmt_override(value, default):
    if value is None:
        return f"({default})"
    # ``ent_coef`` may be a {start, end, schedule} mapping -- render
    # it compactly as e.g. ``0.10->0.03*`` so the table stays one line.
    if isinstance(value, dict):
        return f"{value.get('start', '?')}->{value.get('end', '?')}*"
    return f"{value}*"


for s in cfg.curriculum.stages:
    reward_keys = sorted((s.reward_config or {}).keys())
    reward_str = ", ".join(reward_keys) if reward_keys else "(default)"
    print(
        f"  {s.name:<{name_col_w}s}  {s.opponent:<{opp_col_w}s}  "
        f"{s.promotion_win_rate:>4.0%}  {s.max_timesteps:>10,}  "
        f"{_fmt_override(s.max_steps, cfg.env.max_steps):>10s}  "
        f"{_fmt_override(s.max_turns, cfg.env.max_turns):>10s}  "
        f"{_fmt_override(s.ent_coef, cfg.ppo.ent_coef):>13s}  "
        f"{reward_str}"
    )
print("  (* = per-stage override; (n) = inherits cfg default)")

print(f"\nEval freq:     {cfg.eval.eval_freq:,} env steps")
print(f"n_envs:        {cfg.env.n_envs}")
print(f"Total budget:  {sum(s.max_timesteps for s in cfg.curriculum.stages):,} env steps (worst case)")
print(f"Storage:       {'Google Drive' if USE_GOOGLE_DRIVE else 'Local'}")

## 3b. (Optional) Override config values without editing the YAML

`apply_overrides` returns a deep copy of `cfg` with dotted-key overrides
applied -- handy for sweeping a single hyperparam without forking the
YAML. It walks attribute paths through the dataclass sections, so dict
leaves like `env.reward_config` have to be replaced wholesale (you
can't say `env.reward_config.kill`, because dicts aren't dataclasses).
List leaves like `env.enabled_units` are also replaced wholesale --
the override value is the full new roster, not a delta.

Two examples below: bump kill / damage / capture rewards, and shrink
the available unit roster. Reward magnitudes in `bootstrap.yaml` are
pre-scaled to a 1/100 range (see the comment on `env.reward_config`);
pick override values in that same range to stay in the value head's
fittable region. `enabled_units` applies globally across every
curriculum stage -- there is no per-stage override -- so the same
roster is used for training, sanity eval, and replay videos.

In [ ]:
# from reinforcetactics.rl.config import apply_overrides
#
# # Replace the env.reward_config dict in one shot. Spread the existing
# # config so any keys you don't touch keep their YAML values.
# # Values match the YAML's 1/100 scale -- bumping above ~50 will start
# # to dislocate the value head (watch section 5c's value_loss panel).
# cfg = apply_overrides(cfg, {
#     "env.reward_config": {
#         **cfg.env.reward_config,
#         "kill": 0.5,
#         "damage_scale": 0.01,
#         "capture": 15.0,
#     },
# })
#
# print("kill          =", cfg.env.reward_config.get("kill"))
# print("damage_scale  =", cfg.env.reward_config.get("damage_scale"))
# print("capture       =", cfg.env.reward_config.get("capture"))

# # Override the available unit roster without editing the YAML.
# # ``env.enabled_units`` is a list, so the override replaces it
# # wholesale -- pass the full new roster, not a delta. The roster
# # is global: it applies to training envs, the section-6 sanity
# # eval, and the section-8 replay envs alike. Use the same unit
# # codes as the YAML (e.g. W, M, A); ``cfg.validate()`` doesn't
# # check the contents of ``enabled_units``, so a typo here will
# # surface later as a missing unit type at env-construction time.
# cfg = apply_overrides(cfg, {
#     "env.enabled_units": ["W", "M"],
# })
#
# print("enabled_units =", cfg.env.enabled_units)

# # Override per-stage ``max_turns`` to encourage faster, more decisive
# # play. ``max_turns`` is the hard cap on game length: when the cap is
# # hit the episode ends as a draw (which the reward config penalises
# # at the same magnitude as a loss). Tightening it converts stalling
# # games into losses, pushing the policy toward urgency without
# # rewriting the reward function.
# #
# # ``apply_overrides`` only walks dataclass attributes, so it can't
# # index into ``cfg.curriculum.stages`` (a list). Mutate the stage
# # objects directly. The per-stage table printed in section 3 reads
# # ``s.max_turns`` after this cell runs, so confirm the new values
# # appear there before kicking off section 4. ``run_curriculum``
# # captures the resolved value in each stage's ``config.json`` on
# # disk, so reproducibility is preserved even though the YAML is
# # untouched.
# #
# # Two patterns shown below. Pick one.
#
# # ---- Pattern A: one cap per map (flat) ----
# # Simple and safest. Pulls in the recommended starting points; tune
# # from there based on ``end_reasons["max_turns_draw"]`` in
# # ``eval_results.json`` and ``avg_turns`` in
# # ``bootstrap_results.csv``.
# MAX_TURNS_BY_MAP = {
#     "starter.csv":       15,
#     "beginner.csv":      60,
#     "skirmish.csv":      80,
#     "corner_points.csv": 100,
# }
# for s in cfg.curriculum.stages:
#     for suffix, turns in MAX_TURNS_BY_MAP.items():
#         if s.map_file.endswith(suffix):
#             s.max_turns = turns
#             break
#
# # ---- Pattern B: two-step ramp per map (familiarize -> commit) ----
# # Leaves the early random-opponent stages on the loose YAML cap so
# # the policy gets familiar with new map geometry, then drops to the
# # tighter cap once the scripted-bot ladder begins. Avoids stacking
# # two difficulty axes (less time + smarter opponent) on the same
# # stage transition.
# TURN_RAMP = {
#     "starter.csv":       {"familiarize": None, "commit": 15},   # tiny map; no ramp needed
#     "beginner.csv":      {"familiarize": 100,  "commit": 60},
#     "skirmish.csv":      {"familiarize": 120,  "commit": 80},
#     "corner_points.csv": {"familiarize": 150,  "commit": 100},
# }
# SCRIPTED_OPPONENTS = {"simple", "medium", "advanced", "mixed"}
# for s in cfg.curriculum.stages:
#     for suffix, ramp in TURN_RAMP.items():
#         if not s.map_file.endswith(suffix):
#             continue
#         if s.opponent in SCRIPTED_OPPONENTS:
#             s.max_turns = ramp["commit"]
#         elif ramp["familiarize"] is not None:
#             s.max_turns = ramp["familiarize"]
#         break
#
# # Verify -- the section-3 per-stage table will also show these, but
# # a quick echo here makes the override visible right after the cell
# # runs.
# for s in cfg.curriculum.stages:
#     print(f"  {s.name:32s}  map={Path(s.map_file).stem:14s}  max_turns={s.max_turns}")

# # Drop stages from the curriculum without editing the YAML. Useful for
# # smoke-test runs (small subset of maps), or for skipping the largest
# # maps when running on a shorter budget. ``cfg.curriculum.stages`` is
# # a plain list of ``CurriculumStage`` dataclasses, so filter it in
# # place. Three idioms below; pick one.
# #
# # Caveats:
# #   - Run ``cfg.validate()`` after filtering to catch the edge case
# #     of removing every stage (empty list -> ``run_curriculum`` will
# #     reject the config).
# #   - ``pad_to_size`` auto-resolves from the *remaining* stages
# #     (bootstrap.py). Dropping the largest map means observations
# #     pad to a smaller shape; checkpoints from a full-curriculum run
# #     cannot be loaded into the smaller-padded env (obs-space
# #     mismatch). Don't warm-start a filtered run from an unfiltered
# #     checkpoint.
# #   - Section 6's sanity eval runs against the *new* last stage, not
# #     the original ``corner_points_medium``. Confirm the resulting
# #     checkpoint is appropriate for whatever you do downstream.
#
# from pathlib import Path
#
# # ---- Idiom A: drop all stages for one or more maps ----
# DROP_MAPS = {"corner_points.csv"}   # skip the largest map entirely
# cfg.curriculum.stages = [
#     s for s in cfg.curriculum.stages
#     if Path(s.map_file).name not in DROP_MAPS
# ]
#
# # ---- Idiom B: keep only a whitelist of maps (smoke-test run) ----
# # KEEP_MAPS = {"starter.csv", "beginner.csv"}
# # cfg.curriculum.stages = [
# #     s for s in cfg.curriculum.stages
# #     if Path(s.map_file).name in KEEP_MAPS
# # ]
#
# # ---- Idiom C: drop specific stages by name (finer-grained) ----
# # DROP_STAGES = {
# #     "corner_points_random_20",
# #     "corner_points_simple",
# #     "corner_points_medium",
# # }
# # cfg.curriculum.stages = [
# #     s for s in cfg.curriculum.stages if s.name not in DROP_STAGES
# # ]
#
# cfg.validate()
# print(f"Remaining stages: {len(cfg.curriculum.stages)}")
# for s in cfg.curriculum.stages:
#     print(f"  {s.name:32s}  map={Path(s.map_file).stem}")


## 3c. Build a BC warm-start checkpoint

Behavior cloning (BC) pre-trains the policy on scripted-bot demonstrations so the curriculum starts from a competent generalist instead of a near-uniform random policy. See `docs/bootstrap_lessons_learned.md` for the diagnosis chain that motivates this lever (cumulative training matters more than per-stage training; v30's success was effectively a BC warm-start from a prior PPO run; this section bootstraps the equivalent from scripted bots, no prior RL needed).

`BUILD_BC_WARMSTART = True` (default) trains a fresh BC checkpoint at the start of every run and writes it next to `OUTPUT_DIR/bc_warmstart.zip`. Set to `False` to skip BC (cold start, or reuse whatever `warm_start_path` the config already points at). Set to `"auto"` to build only when `cfg.warm_start_path` is set in the YAML but the file doesn't exist on disk -- the placeholder pattern used by configs like `v33_production_bc_warmstart.yaml` (`warm_start_path: benchmarks/bc/YYYYMMDD_HHMMSS/bc_warmstart.zip`).

After BC builds, this cell:
- Sets `cfg.warm_start_path` to the freshly-built checkpoint in `OUTPUT_DIR`, so the curriculum (section 4) picks it up.
- Rewrites the YAML snapshot already copied into `OUTPUT_DIR` to replace the placeholder line with the resolved path — so the saved config is reproducible and matches the run.

The BC `MaskablePPO` is constructed with `cfg.ppo.policy_kwargs` (features extractor + `net_arch`) so the state dict matches what `set_parameters(exact_match=True)` expects in `bootstrap.py`. Without this the BC checkpoint would have SB3 defaults (`CombinedExtractor` + `[64, 64]` MLP) and the curriculum load would fail with missing-key / size-mismatch errors.

**Constraints (current `imitation.py` infra):**
- Requires `env.action_space_type: multi_discrete`. Use a config such as `configs/ppo/bootstrap_sweep/v33_production_bc_warmstart.yaml` -- it sets this and other prerequisites.
- BC scenarios must share map dimensions (`DemonstrationDataset` stacks observations into one tensor). The default scenario file ships beginner.csv (6x6); v33's curriculum is truncated to same-size maps to match.


In [ ]:
# True (default): always (re)build a BC checkpoint at the start of the
# run. False: skip BC entirely. "auto": build only when cfg.warm_start_path
# is set in the config but the file doesn't exist (placeholder pattern in
# configs like v33).
BUILD_BC_WARMSTART = True

BC_SCENARIOS_PATH = Path("configs/imitation/bc_beginner_warmstart.yaml")
# 60 epochs after run 20260525_011808 (BC_EPOCHS=30) showed
# action_type_acc plateaued at ~78% by epoch 3-4 but full_action_acc
# was still climbing at epoch 30 (8% -> 14% at epoch 8 -> 21% at
# epoch 30). The coordinate heads are the slow learners; the
# action-type classifier hits its information-theoretic ceiling
# fast. 60 epochs lets full_action_acc keep improving toward
# whatever its real ceiling is. Cost is ~3-4 min of BC GPU time,
# trivial vs the multi-hour curriculum run that follows.
BC_EPOCHS = 60
BC_BATCH_SIZE = 128
BC_LEARNING_RATE = 3e-4
BC_SEED = 42
# Per-sample loss weight applied to end_turn demonstrations during
# BC training. Default (None) auto-computes ``n_non_end / n_end``
# to balance the ~10:1 demo imbalance, but run 20260525_011808
# (60 epochs, auto-balance only) showed BC's greedy argmax still
# under-predicts end_turn at eval time: WR=0% vs SimpleBot with
# all 30 episodes hitting max_actions_per_turn=60 each turn and
# truncating at max_steps after 50 game-turns. Forcing the weight
# higher than the auto-balance pushes the argmax toward end_turn
# at the right moments. 30 is roughly 3x the natural ratio; if
# the agent now over-ends turns (single-turn games every time)
# back this off; if still stuck try 50-100. See
# docs/bootstrap_lessons_learned.md Failure mode G + the
# never-end-turn attractor diagnosis.
BC_END_TURN_WEIGHT = 30.0


def _decide_bc_build(setting, warm_start_path):
    if setting is True:
        return True, "forced (BUILD_BC_WARMSTART=True)"
    if setting is False:
        return False, "disabled (BUILD_BC_WARMSTART=False)"
    # "auto": build only when the config asks for a warm-start that
    # doesn't exist yet -- the placeholder pattern. An existing
    # checkpoint is reused; an unset warm_start_path means cold start.
    if not warm_start_path:
        return False, "auto: cfg.warm_start_path is not set (cold start)"
    if Path(warm_start_path).exists():
        return False, f"auto: existing checkpoint at {warm_start_path}"
    return True, f"auto: warm_start_path '{warm_start_path}' does not exist -- building"


build_bc, bc_reason = _decide_bc_build(BUILD_BC_WARMSTART, cfg.warm_start_path)
print(f"BC decision: {'BUILD' if build_bc else 'SKIP'} -- {bc_reason}")

if build_bc:
    import re

    from reinforcetactics.rl import (
        load_scenarios_from_yaml,
        make_maskable_env,
        make_warm_started_model,
    )
    from reinforcetactics.rl.bootstrap import _resolve_policy_kwargs

    if cfg.env.action_space_type != "multi_discrete":
        raise RuntimeError(
            f"BC warm-start requires action_space_type=multi_discrete; "
            f"loaded config has '{cfg.env.action_space_type}'. "
            f"Use a config such as configs/ppo/bootstrap_sweep/"
            f"v33_production_bc_warmstart.yaml."
        )

    scenarios = load_scenarios_from_yaml(BC_SCENARIOS_PATH)
    print(f"BC: loaded {len(scenarios)} scenarios from {BC_SCENARIOS_PATH}")
    for sc in scenarios:
        print(
            f"  - {sc.name}: {sc.demonstrator} vs {sc.opponent}, "
            f"map={sc.map_file}, units={sc.enabled_units}, "
            f"episodes={sc.n_episodes}, weight={sc.weight}"
        )

    # Template env mirrors the spaces the curriculum will use. The
    # MaskablePPO model BC trains is bound to this env's obs/action shapes
    # so the checkpoint loads cleanly later via warm_start_path.
    first_stage = cfg.curriculum.stages[0]
    template_env = make_maskable_env(
        map_file=first_stage.map_file,
        opponent="medium",  # arbitrary -- only used for shape resolution
        action_space_type=cfg.env.action_space_type,
        enabled_units=cfg.env.enabled_units,
        max_turns=first_stage.resolve_max_turns(cfg.env),
    )

    # Forward the curriculum's policy_kwargs (features extractor +
    # net_arch) into the BC MaskablePPO constructor so the checkpoint
    # state dict matches what bootstrap.py loads via
    # set_parameters(exact_match=True). Without this the BC model is
    # built with SB3 defaults (CombinedExtractor + [64, 64] MLP) and
    # the downstream load fails with "Missing keys ...
    # features_extractor.cnn.*" plus a 64-vs-256 size mismatch on the
    # MLP heads. Mirrors what scripts/build_bc_warmstart.py does.
    bc_policy_kwargs = _resolve_policy_kwargs(cfg.ppo.policy_kwargs)
    bc_ppo_kwargs = {"verbose": 0}
    if bc_policy_kwargs:
        bc_ppo_kwargs["policy_kwargs"] = bc_policy_kwargs
        fe = bc_policy_kwargs.get("features_extractor_class")
        fe_name = getattr(fe, "__name__", str(fe)) if fe else "(SB3 default)"
        print(f"BC: features_extractor={fe_name}  net_arch={bc_policy_kwargs.get('net_arch')}")

    bc_checkpoint = OUTPUT_DIR / "bc_warmstart.zip"
    print(
        f"BC: training {BC_EPOCHS} epochs (batch={BC_BATCH_SIZE}, lr={BC_LEARNING_RATE}, end_turn_weight={BC_END_TURN_WEIGHT})"
    )
    print(f"BC: output -> {bc_checkpoint}")

    bc_model, bc_dataset, bc_stats = make_warm_started_model(
        env=template_env,
        n_epochs=BC_EPOCHS,
        batch_size=BC_BATCH_SIZE,
        learning_rate=BC_LEARNING_RATE,
        seed=BC_SEED,
        ppo_kwargs=bc_ppo_kwargs,
        scenarios=scenarios,
        end_turn_weight=BC_END_TURN_WEIGHT,
    )

    bc_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    bc_model.save(str(bc_checkpoint))

    if bc_stats:
        final = bc_stats[-1]
        print(
            f"BC: done -- demos={len(bc_dataset):,}  "
            f"final_loss={final.loss:.4f}  "
            f"action_type_acc={final.accuracy_action_type:.3f}  "
            f"full_action_acc={final.accuracy_full:.3f}"
        )

    # Point the curriculum at the freshly-built checkpoint. bootstrap.py
    # reads cfg.warm_start_path before stage 1 and calls set_parameters,
    # so the curriculum train below picks up these weights automatically.
    cfg.warm_start_path = str(bc_checkpoint)
    print(f"BC: cfg.warm_start_path -> {cfg.warm_start_path}")

    # Rewrite the YAML snapshot copied into OUTPUT_DIR at section 3 so
    # the saved config matches what actually ran. The original copy
    # still has the YYYYMMDD_HHMMSS placeholder from the source YAML;
    # without this rewrite the snapshot would be unreproducible (the
    # placeholder path doesn't resolve, so reloading the snapshot would
    # either fail or cold-start). Line-based replace preserves the
    # file's comments and key ordering, unlike yaml.safe_dump which
    # would strip both.
    snapshot_yaml = OUTPUT_DIR / CONFIG_PATH.name
    if snapshot_yaml.exists():
        text = snapshot_yaml.read_text()
        new_line = f"warm_start_path: {cfg.warm_start_path}\n"
        pattern = re.compile(r"^warm_start_path:.*\n", re.MULTILINE)
        if pattern.search(text):
            text = pattern.sub(new_line, text, count=1)
            print(f"BC: updated warm_start_path in {snapshot_yaml}")
        else:
            text = new_line + text
            print(f"BC: prepended warm_start_path to {snapshot_yaml}")
        snapshot_yaml.write_text(text)
    else:
        print(f"BC: snapshot YAML not found at {snapshot_yaml}; skipped rewrite")
else:
    print("BC: nothing to do.")

## 3d. BC diagnostics

Two views into the warm-start that section 3c just produced:

- **Per-scenario demonstrator outcomes** — for each scenario in `BC_SCENARIOS_PATH`, how often the demonstrator (the policy we're cloning) actually won, how many turns the games took, and the action-mix balance. A scenario with demonstrator WR < 50% is teaching BC losing trajectories at the majority rate; long avg turns near `max_turns` means games are timing out as draws instead of resolving.

- **BC training curves** — masked cross-entropy loss falling and `action_type` accuracy rising across epochs. Flat curves mean BC isn't learning; a falling loss with stalled accuracy usually means class imbalance (see the auto-balance for `end_turn` in `imitation.py`).

Both panels save under `OUTPUT_DIR/charts/bc/`.

In [ ]:
import json

from reinforcetactics.rl import format_scenario_stats_table
from reinforcetactics.rl.viz import plot_bc_demo_outcomes, plot_bc_training_curves

# Guard for standalone re-runs. ``build_bc`` is bound in section 3c;
# if the kernel has been restarted and the user is re-running just
# this cell (common when iterating on plot styling), 3c hasn't run
# this session and a bare ``not build_bc`` raises NameError. The
# ``in globals()`` test makes the cell a clean no-op in that case,
# mirroring the no-op behaviour when 3c ran with
# BUILD_BC_WARMSTART = False.
if "build_bc" not in globals() or not build_bc:
    print("BC diagnostics: skipped (run section 3c first to build a BC checkpoint).")
else:
    bc_charts_dir = CHARTS_DIR / "bc"
    bc_charts_dir.mkdir(parents=True, exist_ok=True)

    # Snapshot the BC hyperparameters once so every downstream JSON
    # carries the same config block. Two runs with different
    # ``BC_EPOCHS`` (or batch size, lr, seed) produce visually-similar
    # outputs; without an explicit record the user has to remember
    # which value they used when comparing checkpoints. Keep it
    # alongside the per-epoch / per-scenario data rather than in a
    # separate file so a single ``cat`` shows the full context.
    bc_config = {
        "n_epochs": BC_EPOCHS,
        "batch_size": BC_BATCH_SIZE,
        "learning_rate": BC_LEARNING_RATE,
        "seed": BC_SEED,
        "end_turn_weight": BC_END_TURN_WEIGHT,
        "scenarios_path": str(BC_SCENARIOS_PATH),
        "checkpoint_path": str(bc_checkpoint),
    }

    if bc_dataset.scenario_stats:
        table = format_scenario_stats_table(bc_dataset.scenario_stats)
        print("BC: per-scenario demonstrator outcomes")
        print(table)

        # Persist the text table to Drive so it survives the Colab
        # runtime ending. Stdout-only output is gone the moment the
        # kernel detaches; the file lands next to the BC charts.
        stats_txt = bc_charts_dir / "scenario_stats.txt"
        stats_txt.write_text(table + "\n")
        print(f"\nBC: scenario stats text -> {stats_txt}")

        # Structured dump (per-scenario records + the computed
        # properties used in the chart) so downstream notebooks /
        # scripts can re-analyse BC demo quality without rebuilding
        # the warmstart. JSON because dataclasses.asdict + json.dumps
        # is enough -- no need for a heavier serialiser here.
        #
        # ``step_budget_exhausted`` is included so action-loop
        # timeouts (bots stuck in an intra-turn loop) are visible
        # separately from genuine map-cap draws.
        records = []
        for s in bc_dataset.scenario_stats:
            records.append(
                {
                    "name": s.name,
                    "demonstrator": s.demonstrator,
                    "opponent": s.opponent,
                    "map_file": s.map_file,
                    "n_episodes": s.n_episodes,
                    "demo_wins": s.demo_wins,
                    "demo_losses": s.demo_losses,
                    "draws": s.draws,
                    "step_budget_exhausted": s.step_budget_exhausted,
                    "total_games": s.total_games,
                    "demo_win_rate": s.demo_win_rate,
                    "draw_rate": s.draw_rate,
                    "avg_turns": s.avg_turns,
                    "avg_demos_per_game": s.avg_demos_per_game,
                    "total_demos": s.total_demos,
                    "total_turns": s.total_turns,
                    "end_reasons": dict(s.end_reasons),
                }
            )
        stats_json = bc_charts_dir / "scenario_stats.json"
        stats_json.write_text(json.dumps({"bc_config": bc_config, "scenarios": records}, indent=2))
        print(f"BC: scenario stats json -> {stats_json}")
    else:
        print("BC: scenario_stats not populated (collect_demonstrations / make_warm_started_model are the only producers).")

    if bc_stats:
        plot_bc_training_curves(bc_stats, charts_dir=bc_charts_dir)
        plt.show()
        # Per-epoch supervised metrics dumped to JSON so the training
        # curves can be re-rendered or compared across BC builds
        # without re-running the BC pipeline (which is the expensive
        # part of section 3c). The ``bc_config`` block makes runs
        # with different ``BC_EPOCHS`` / batch / lr / seed
        # immediately distinguishable -- previously you had to count
        # the per-epoch records to infer how many epochs were run.
        bc_stats_json = bc_charts_dir / "bc_training_stats.json"
        bc_stats_json.write_text(
            json.dumps(
                {
                    "bc_config": bc_config,
                    "epochs": [
                        {
                            "epoch": s.epoch,
                            "loss": s.loss,
                            "accuracy_action_type": s.accuracy_action_type,
                            "accuracy_full": s.accuracy_full,
                        }
                        for s in bc_stats
                    ],
                },
                indent=2,
            )
        )
        print(f"\nBC: training stats json -> {bc_stats_json}")
    if bc_dataset.scenario_stats:
        plot_bc_demo_outcomes(bc_dataset.scenario_stats, charts_dir=bc_charts_dir)
        plt.show()

## 3e. BC sanity eval: how does the warm-start actually play?

Section 3d's stats describe BC's *training data* (demonstrator quality) and *supervised metrics* (loss / accuracy), but neither tells you how the resulting policy actually plays. A BC checkpoint can have 76% action_type accuracy and still lose every game because the coordinate predictions are mostly wrong.

This cell runs the BC checkpoint against `SimpleBot`, `MediumBot`, and `AdvancedBot` on the first curriculum stage's map for ~30 episodes each, *before* PPO training. Three numbers, ~1-2 minutes total:

- **WR vs SimpleBot**: pure sanity check. Should be >80%. If it's <40%, BC is broken (obs/mask mismatch, wrong architecture, etc.) and PPO won't fix it.
- **WR vs MediumBot**: the realistic BC ceiling. The demonstrator beat MediumBot ~57% in BC scenarios, so BC should land somewhere in the 40-70% range.
- **WR vs AdvancedBot**: the gap PPO has to close. Often 0-20%; that's expected.

Results are saved to `OUTPUT_DIR/charts/bc/bc_sanity_eval.json` so you can compare across BC builds without re-evaluating.

**How to interpret each PPO eval afterwards**: section 4's eval @ 8 (first eval, essentially BC-with-1-update) should land near the corresponding `bc_sanity_eval` WR for the stage's opponent. If PPO's WR is materially *lower* than the BC baseline mid-training, PPO is hurting the policy — investigate before raising the budget.

In [ ]:
import json

from reinforcetactics.rl import evaluate_bc_against_bot_ladder

# Guard for standalone re-runs. ``bc_model`` is bound in section 3c
# only when a build ran this session; if the kernel restarted and 3c
# wasn't re-executed, this cell would NameError on ``bc_model``.
# Mirrors the section 3d guard pattern.
if "build_bc" not in globals() or not build_bc:
    print("BC sanity eval: skipped (run section 3c first to build a BC checkpoint).")
else:
    # ``evaluate_bc_against_bot_ladder`` lives in
    # ``reinforcetactics.rl.imitation`` and uses ``make_stage_env``
    # internally, so the eval env always matches the production
    # curriculum env (reward_config, max_actions_per_turn, scale
    # factors, etc.). The previous inline implementation here got
    # bitten by omitting ``reward_config`` and ``max_actions_per_turn``
    # in run 20260524_225835 (reward -30k / W/L/D 0/0/30 was a
    # measurement artifact, not a BC failure -- see
    # docs/bootstrap_lessons_learned.md Failure mode G). Routing
    # through the library helper kills that bug class permanently.
    first_stage = cfg.curriculum.stages[0]
    print("BC sanity eval: how does the warm-start play before PPO?")
    print(
        f"  map={first_stage.map_file}, max_turns={first_stage.resolve_max_turns(cfg.env)}, "
        f"max_actions_per_turn={cfg.env.max_actions_per_turn}, n_episodes=30 per opponent"
    )
    print()

    sanity_metrics = evaluate_bc_against_bot_ladder(bc_model, cfg, n_episodes=30)

    # Distil ``evaluate_model``'s metrics dict to the fields we
    # actually want printed / persisted -- the full dict carries a
    # lot of stuff (per-episode rewards/lengths arrays, combat
    # breakdowns) that bloats the JSON without helping diagnosis.
    sanity_results = {}
    for opp, m in sanity_metrics.items():
        sanity_results[opp] = {
            "win_rate": m["win_rate"],
            "avg_reward": m["avg_reward"],
            "avg_length": m["avg_length"],
            "avg_turns": m["avg_turns"],
            "wins": m["wins"],
            "losses": m["losses"],
            "draws": m["draws"],
            "end_reasons": dict(m.get("end_reasons", {})),
        }
        # Print includes len + turns + end_reasons so the
        # never-end-turn / max_steps_truncate failure mode is visible
        # directly (avg_turns near 0 with avg_length near max_steps =
        # the agent is looping without ending turns).
        end_reason_summary = ", ".join(f"{k}={v}" for k, v in m.get("end_reasons", {}).items() if v > 0) or "(none)"
        print(
            f"  BC vs {opp:10s}  WR={m['win_rate']:5.1%}  reward={m['avg_reward']:+8.1f}  "
            f"len={m['avg_length']:5.0f}  turns={m['avg_turns']:5.1f}  "
            f"W/L/D={m['wins']}/{m['losses']}/{m['draws']}  end={end_reason_summary}"
        )

    # Carry the BC hyperparameters into the sanity-eval JSON so an
    # eval result can be traced back to the exact training config
    # that produced it. Without this, comparing two
    # ``bc_sanity_eval.json`` files (e.g. BC_EPOCHS=8 vs
    # BC_EPOCHS=30 runs) requires hunting the matching
    # bc_training_stats.json to recover the config. Same block as
    # section 3d so a single ``diff`` between two runs' sanity JSONs
    # shows the config delta inline.
    bc_config = {
        "n_epochs": BC_EPOCHS,
        "batch_size": BC_BATCH_SIZE,
        "learning_rate": BC_LEARNING_RATE,
        "seed": BC_SEED,
        "end_turn_weight": BC_END_TURN_WEIGHT,
        "scenarios_path": str(BC_SCENARIOS_PATH),
        "checkpoint_path": str(bc_checkpoint),
    }

    bc_charts_dir = CHARTS_DIR / "bc"
    bc_charts_dir.mkdir(parents=True, exist_ok=True)
    sanity_json = bc_charts_dir / "bc_sanity_eval.json"
    sanity_json.write_text(
        json.dumps(
            {
                "bc_config": bc_config,
                "map_file": first_stage.map_file,
                "max_turns": first_stage.resolve_max_turns(cfg.env),
                "max_steps": first_stage.resolve_max_steps(cfg.env),
                "max_actions_per_turn": cfg.env.max_actions_per_turn,
                "n_episodes_per_opponent": 30,
                "results": sanity_results,
            },
            indent=2,
        )
    )
    print(f"\nBC: sanity eval saved -> {sanity_json}")

## 4. Run the curriculum

One `model.learn()` call per stage, with the env swapped between stages
via `model.set_env(...)`. The shared `MaskablePPO` instance survives the
whole run, so the policy carries forward; only the `(map, opponent)` pair
changes. Best-by-eval-win-rate checkpoints land in
`OUTPUT_DIR/<stage_name>/best_model.zip`; the post-promotion model is
written as `stage_final.zip`.

In [ ]:
try:
    result = run_curriculum(cfg, output_dir=OUTPUT_DIR)
    stalled = None
except CurriculumStalled as exc:
    print(f"\nSTALLED: {exc}")
    print("Investigate before raising the budget — a stuck stage usually")
    print("signals reward shaping or hyperparam issues, not insufficient steps.")
    # Recover the partial run -- history through the stalled stage,
    # plus a final_model.zip saved at the moment of stall -- so the
    # rest of the notebook (diagnostics, checkpoint snapshot, replay
    # videos) still runs end-to-end. Without this, a stall would
    # discard every artifact you'd want for triage.
    result = exc.partial_result()
    stalled = exc

if result is not None:
    if result.get("stalled"):
        print(f"\nPartial run available (stalled at '{result['stalled_stage']}'):")
        if result.get("final_model_path"):
            print(f"Snapshot at stall: {result['final_model_path']}")
    else:
        print(f"\nFinal model: {result['final_model_path']}")
    for h in result["history"]:
        last = h["results"][-1] if h["results"] else None
        last_wr = f"{last['win_rate']:.1%}" if last else "n/a"
        print(f"  {h['stage']:18s}  promoted={h['promoted']!s:5s}  best_wr={h['best_win_rate']:.1%}  last_wr={last_wr}")

## 4b. Snapshot per-stage checkpoints

`run_curriculum` writes three files into `OUTPUT_DIR/<stage_name>/`
for each stage as soon as the stage finishes — `stage_final.zip`
(post-promotion model), `best_model.zip` (best by eval win-rate
during the stage), and `config.json` (resolved env + reward + PPO
hyperparams + seed + git/library/hardware metadata). Writing
`config.json` inside the runner means a Colab disconnect or a later
`CurriculumStalled` on a subsequent stage still leaves every
completed stage with a reproducible config snapshot on disk.

This cell mirrors the two checkpoint zips into a flat
`OUTPUT_DIR/checkpoints/` directory so every stage's policy is easy
to reload later — useful for replaying a specific (map, opponent)
combination or comparing intermediate policies side-by-side without
walking the per-stage subfolders.

In [ ]:
import shutil

if result is None:
    stage_checkpoints = {}
    print("Skipped — no successful run to snapshot.")
else:
    checkpoints_dir = OUTPUT_DIR / "checkpoints"
    checkpoints_dir.mkdir(parents=True, exist_ok=True)

    # One row per stage with the canonical paths the rest of the
    # notebook (sanity eval, replay video) consumes. Prefer
    # ``best_model.zip`` when it's strictly better than the post-stage
    # snapshot — it's the highest eval-WR checkpoint within the stage.
    # ``run_curriculum`` already wrote ``config.json`` next to each
    # stage's checkpoints; we don't re-emit it here.
    stage_checkpoints = {}
    for h in result["history"]:
        stage_name = h["stage"]
        stage_dir = OUTPUT_DIR / stage_name
        stage_final = stage_dir / "stage_final.zip"
        best_model = stage_dir / "best_model.zip"

        # Copy stage_final under the stage's name; copy best_model with
        # a `_best` suffix when present. Both stay paired with the
        # source folder so the per-stage subdirs remain authoritative.
        flat_final = checkpoints_dir / f"{stage_name}.zip"
        if stage_final.exists():
            shutil.copy2(stage_final, flat_final)
        flat_best = checkpoints_dir / f"{stage_name}_best.zip"
        if best_model.exists():
            shutil.copy2(best_model, flat_best)

        stage_checkpoints[stage_name] = {
            "map_file": next(s.map_file for s in cfg.curriculum.stages if s.name == stage_name),
            "opponent": next(s.opponent for s in cfg.curriculum.stages if s.name == stage_name),
            "stage_final": str(flat_final) if flat_final.exists() else None,
            "best_model": str(flat_best) if flat_best.exists() else None,
            "promoted": h["promoted"],
            "best_win_rate": h["best_win_rate"],
        }

    print(f"Snapshotted {len(stage_checkpoints)} stage checkpoints to:")
    print(f"  {checkpoints_dir}")
    print()
    for name, meta in stage_checkpoints.items():
        promo = "promoted" if meta["promoted"] else "stalled"
        final_path = Path(meta["stage_final"]).name if meta["stage_final"] else "(missing)"
        best_path = Path(meta["best_model"]).name if meta["best_model"] else "(no best)"
        print(f"  {name:30s}  {promo:8s}  best_wr={meta['best_win_rate']:5.1%}  final={final_path}  best={best_path}")

## 5. Win-rate over the full curriculum

Concatenates eval snapshots across stages onto a single timestep axis.
Vertical lines mark stage transitions; horizontal lines mark each stage's
promotion threshold. A regression bump right after a transition is
expected with hard cutoffs — the policy is encountering a new opponent
for the first time.

In [ ]:
from reinforcetactics.rl.viz import plot_curriculum_summary

if result is None:
    print("Skipped — no successful run to plot.")
else:
    fig = plot_curriculum_summary(
        result["history"],
        cfg.curriculum.stages,
        charts_dir=CHARTS_DIR,
    )
    if fig is not None:
        plt.show()

## 5a. Curriculum-wide metric timelines

Companion to section 5: same flatten-across-stages approach, but for
the diagnostic metrics that section 5b otherwise only shows *per
stage*. Each chart concatenates every stage's eval snapshots onto a
single cumulative-timestep axis, with vertical dashed lines at stage
transitions, so it's easy to see how a metric like unit-build mix or
action distribution evolves as the curriculum shifts the (map,
opponent) pair.

Six panels rendered separately (one per metric) so each retains its
own legend and y-axis scale:

- **Episode length** — climbing then dropping as the policy learns
  to win earlier; saturating at the cap is the timeout-into-draw mode.
- **Outcome × end-reason** — wins via HQ capture (intended) vs
  elimination (attrition); draws timing out at max turns.
- **Reward decomposition** — growing orange ``terminal`` band over
  the whole curriculum is the cleanest sign the policy is converging
  to actually winning, not just farming shaping rewards.
- **Action distribution** — surfaces curriculum-wide collapse modes
  (e.g. ``end_turn`` spam) that a per-stage view might miss when each
  stage's local trajectory looks fine.
- **Unit-build distribution** — whether the agent's army composition
  diversifies as opponents toughen, or stays a one-trick stack.
- **Combat summary** — per-game captures / kills / damage delta /
  attack-seize activity across the whole run.

Files land under ``OUTPUT_DIR/charts/curriculum/`` as
``curriculum_<metric>.png`` so they don't collide with the per-stage
PNGs written by section 5b.


In [ ]:
from reinforcetactics.rl.viz import plot_curriculum_metrics

if result is None or not result.get("history"):
    print("Skipped — no eval history to plot.")
else:
    figures = plot_curriculum_metrics(
        result["history"],
        charts_dir=CHARTS_DIR,
    )
    for fig in figures.values():
        if fig is not None:
            plt.show()
    saved = [name for name, fig in figures.items() if fig is not None]
    if saved:
        print(f"\nCurriculum-wide charts saved to {CHARTS_DIR / 'curriculum'}")
        for name in saved:
            print(f"  curriculum_{name}.png")

## 5b. Stage diagnostics: outcomes / length / reward mix / action mix

Four panels per stage (drawn separately so each stage's timestep axis
is locally meaningful):

1. **W/L/D over time** — `0/0/30` (all draws) means episodes are timing
   out without a winner; `0/30/0` (all losses) means the opponent is
   crushing the agent. Different fixes for each.
2. **Mean episode length** — saturating at `max_steps` confirms the
   turn cap is the rate-limiter; climbing then dropping is the policy
   learning to win earlier.
3. **Reward decomposition** — large green `action` band with ~0
   `terminal` band reads as 'farms shaping but never wins'. Growing
   orange `terminal` band is healthy convergence.
4. **Action mix** — dominant `end_turn` is policy collapse;
   `create_unit`-heavy with no `attack`/`seize` is over-economy.

In [ ]:
from reinforcetactics.rl.viz import plot_stage_diagnostics

if result is None:
    print("Skipped — no successful run to diagnose.")
else:
    for h in result["history"]:
        if not h["results"]:
            print(f"[{h['stage']}] no eval snapshots, skipping diagnostics")
            continue
        # One subdirectory per stage under the run-level ``charts/``
        # root. ``plot_stage_diagnostics`` writes the canonical
        # filenames (episode_length.png, outcome_breakdown.png, …) so
        # both notebooks produce the same artefacts; the bootstrap
        # adds a per-stage namespace, the training notebook calls it
        # once at the run root.
        stage_charts_dir = CHARTS_DIR / h["stage"]
        stage_charts_dir.mkdir(parents=True, exist_ok=True)
        figures = plot_stage_diagnostics(
            h["results"],
            charts_dir=stage_charts_dir,
            title_suffix=h["stage"],
        )
        for fig in figures.values():
            if fig is not None:
                plt.show()
        print(f"[{h['stage']}] diagnostics saved to {stage_charts_dir}")

## 5c. PPO update health across the curriculum

Six panels on a shared timestep axis spanning the entire bootstrap
run: top row is the per-eval scalars (win rate, avg reward, episode
length); bottom row is the per-rollout PPO internals (`approx_kl`,
`explained_variance`, `value_loss`). Vertical dashed lines mark
stage transitions so you can see whether the value head holds up
across map / opponent shifts or has to re-fit from scratch.

**What to look for:**

- **`approx_kl` near zero across a stage** — policy isn't actually
  updating. Usually means the gradient is being absorbed by value-head
  fitting; check the next two panels.
- **`explained_variance` flat near zero or negative** — value head
  isn't fitting the returns. With reward magnitudes in the thousands
  this is the most likely failure mode of large-scale shaping.
- **`value_loss` growing without bound** (especially on log scale) —
  value targets are outpacing the head's capacity / lr. Consider
  scaling all reward weights down by 10–100x or adding `clip_range_vf`.
- **Sharp `value_loss` spike at a stage transition that decays back
  down** — healthy: the value head is re-fitting the new reward scale.
  A spike that *stays* elevated is the diagnosis pattern for the
  "catastrophic value dislocation" mentioned in the bootstrap config
  comments.

In [ ]:
from reinforcetactics.rl.viz import plot_eval_curves

if result is None or not result.get("history"):
    print("Skipped — no eval history.")
else:
    metrics_callback = result.get("metrics_callback")
    train_records = metrics_callback.records if metrics_callback is not None else None

    # Concatenate per-stage eval results onto the cumulative timestep
    # axis. ``plot_eval_curves`` expects a single flat list; the
    # per-stage results already carry cumulative timesteps because the
    # runner uses ``reset_num_timesteps=False``.
    all_results = []
    for h in result["history"]:
        all_results.extend(h["results"])

    # Stage-transition x-coordinates: take the last eval timestep of
    # each promoted stage (excluding the final stage, which doesn't
    # transition to anything). Vertical dashed lines on every panel.
    stage_boundaries = []
    for h in result["history"][:-1]:
        if h["results"]:
            stage_boundaries.append(h["results"][-1]["timesteps"])

    if not all_results:
        print("Skipped — no eval results across stages.")
    else:
        plot_eval_curves(
            all_results,
            train_records=train_records,
            opponent_label="curriculum",
            charts_dir=CHARTS_DIR,
            stage_boundaries=stage_boundaries,
        )
        plt.show()

## 6. Sanity eval on the final stage

Reload the saved checkpoint and run a fresh evaluation against the
hardest stage (beginner map vs medium bot) to confirm the exit criterion
really holds. Uses more episodes than the in-training eval for a tighter
estimate.

In [ ]:
if result is None or not result.get("history") or not result.get("final_model_path"):
    print("Skipped — no final checkpoint to sanity-eval.")
else:
    # On a successful run this is ``cfg.curriculum.stages[-1]``. On a
    # stall we never reached the configured final stage, so eval
    # against the last stage we *did* train on -- the same one the
    # saved ``final_model.zip`` was last optimised against -- so the
    # numbers are interpretable.
    last_stage_name = result["history"][-1]["stage"]
    final_stage = next(s for s in cfg.curriculum.stages if s.name == last_stage_name)
    sanity_env = make_maskable_env(
        map_file=final_stage.map_file,
        opponent=final_stage.opponent,
        max_steps=final_stage.resolve_max_steps(cfg.env),
        max_turns=final_stage.resolve_max_turns(cfg.env),
        reward_config=final_stage.resolve_reward_config(cfg.env),
        enabled_units=cfg.env.enabled_units,
        action_space_type=cfg.env.action_space_type,
        seed=cfg.seed + 9999,  # different seed than training eval
        opponent_kwargs=final_stage.opponent_kwargs,
        # When the curriculum spans maps of different sizes, run_curriculum
        # resolves a curriculum-wide pad shape and writes it back onto
        # cfg.env.pad_to_size. The saved model was trained on those padded
        # observations, so the sanity env has to forward the same value or
        # MaskablePPO.load rejects the smaller raw obs shape.
        pad_to_size=cfg.env.pad_to_size,
    )
    loaded = MaskablePPO.load(result["final_model_path"])
    metrics = evaluate_model(loaded, sanity_env, n_episodes=100, seed=cfg.seed + 9999)
    sanity_env.close()
    label = f"{final_stage.name}{' [stalled]' if result.get('stalled') else ''}"
    print(
        f"Sanity eval ({label}, n=100):  "
        f"WR={metrics['win_rate']:.1%}  "
        f"reward={metrics['avg_reward']:+.1f}  "
        f"W/L/D={metrics['wins']}/{metrics['losses']}/{metrics['draws']}"
    )
    if metrics["win_rate"] < final_stage.promotion_win_rate:
        print(
            f"  WARNING: sanity WR {metrics['win_rate']:.1%} < threshold "
            f"{final_stage.promotion_win_rate:.1%}. The policy may be\n"
            "  overfit to the training eval seed. Consider a longer final\n"
            "  stage or rolling-window promotion before warm-starting self-play."
        )

## 7. Hand off to self-play

The final checkpoint is the artifact `notebooks/ppo_training.ipynb`'s
self-play setup should load. Drop this snippet into that notebook in
place of the fresh `MaskablePPO(...)` constructor call:

```python
from sb3_contrib import MaskablePPO

BOOTSTRAP_CHECKPOINT = '<paste path printed below>'
model = MaskablePPO.load(BOOTSTRAP_CHECKPOINT, env=self_play_vec_env)
```

`set_env()` is called implicitly by `MaskablePPO.load(..., env=...)`, so
the policy weights load while the env is the new self-play one.

In [ ]:
if result is None or not result.get("final_model_path"):
    print("No checkpoint to hand off.")
elif result.get("stalled"):
    print(f"PARTIAL hand-off only — curriculum stalled at '{result['stalled_stage']}'.")
    print("This checkpoint was NOT trained against the configured final stage,")
    print("so don't warm-start self-play with it without first fixing the stall and re-running.")
    print(f"  {result['final_model_path']}")
else:
    print("Hand-off path:")
    print(f"  {result['final_model_path']}")
    print("\nIn ppo_training.ipynb:")
    print(f"  BOOTSTRAP_CHECKPOINT = '{result['final_model_path']}'")
    print("  model = MaskablePPO.load(BOOTSTRAP_CHECKPOINT, env=self_play_vec_env)")

## 8. Replay videos per (map, opponent)

Loads each stage's snapshotted checkpoint (preferring the best-model
copy when present) and records one full episode against that stage's
opponent on that stage's map. Output goes to `OUTPUT_DIR/videos/` as
`<stage_name>.mp4` plus a paired `.json` replay. Useful for eyeballing
*what* the policy actually learned at each curriculum step — win-rate
curves tell you whether it works, the videos tell you how it works.

Skips stages whose checkpoint is missing. If `imageio_ffmpeg` isn't
installed the writer falls back to OpenCV's `mp4v` codec (may not
play in QuickTime / Preview but is fine in browser previews).

In [ ]:
from reinforcetactics.rl import record_curriculum_replays

if result is None or not stage_checkpoints:
    print("Skipped — no checkpoints to replay.")
    video_summary = []
else:
    videos_dir = OUTPUT_DIR / "videos"
    # ``record_curriculum_replays`` lives in
    # ``reinforcetactics.rl.bootstrap`` and uses ``make_stage_env``
    # internally, so each replay env mirrors the stage's production
    # env down to ``reward_config`` and ``max_actions_per_turn``.
    # Previously the inline loop hand-coded the env construction; if
    # the production env grew a new kwarg, the replay env would
    # silently drift. Routing through the helper keeps them aligned.
    video_summary = record_curriculum_replays(
        cfg,
        stage_checkpoints,
        videos_dir,
    )
    print()
    print(f"Saved {len(video_summary)} replay video(s) to {videos_dir}")

## 9. Per-stage individual game statistics (2x3)

For each replay recorded above, plot a 2x3 panel of per-step
diagnostics. Useful for asking "what did the agent *actually* do this
game?" — independent of the aggregate eval curves in section 5/5b.

- **Top row (per player over time):** unit count, gold, structures
  owned (towers + buildings + HQ).
- **Bottom-left:** action mix used in this game — `create_unit` is
  split by which type was actually spawned (`create_W`, `create_M`, …)
  so you can see e.g. "the agent only ever built Warriors". Blue bars
  = creates, green bars = other actions.
- **Bottom-middle:** cumulative reward by component (`action`,
  `shaping_delta`, `invalid_penalty`, `terminal`).
- **Bottom-right:** outcome summary — winner, end reason
  (`hq_capture` / `elimination` / `max_turns_draw` /
  `max_steps_truncate`), final unit/gold/structure counts, and
  per-type creation counts.

Each panel reads from the `step_stats` returned by
`record_evaluation_to_video`. One file per stage lands at
`OUTPUT_DIR/charts/individual_game_stats/individual_game_stats_<stage>.png`,
matching the `individual_game_stats_best` / `_final` filenames that
`ppo_training.ipynb` 9e produces — so glob patterns over `charts/`
work the same way for both notebooks.

In [ ]:
from reinforcetactics.rl.viz import plot_individual_game_stats

if not video_summary:
    print("Skipped — no recorded games to plot. Run section 8 first.")
else:
    stats_dir = CHARTS_DIR / "individual_game_stats"
    stats_dir.mkdir(parents=True, exist_ok=True)
    for entry in video_summary:
        if not entry.get("step_stats"):
            print(f"[{entry['stage']}] no step_stats on entry, skipping")
            continue
        # Pass the stage as ``title_suffix`` and let the helper produce
        # its canonical ``individual_game_stats_<stage>.png`` filename
        # (matching the ``individual_game_stats_best.png`` /
        # ``individual_game_stats_final.png`` pair that
        # ppo_training.ipynb 9e produces). Keeping the filename scheme
        # identical across notebooks lets downstream tooling glob a
        # single pattern instead of special-casing per notebook.
        plot_individual_game_stats(
            entry,
            charts_dir=stats_dir,
            title_suffix=entry["stage"],
        )
        plt.show()
    print(f"\nSaved per-stage stats panels to {stats_dir}")

In [ ]:
import time

print("Training finished. Disconnecting runtime in 5 seconds...")
time.sleep(5)

from google.colab import runtime

runtime.unassign()